# Phase 2: Preprocessing and DataLoaders

In this phase we prepare the images for PyTorch training. We will resize images to 224 x 224, split the dataset into train/validation/test sets, apply simple augmentation only to training images, and verify tensor shapes.

Why this matters: a model cannot train directly from image files. PyTorch needs tensors, labels, and batches.

## 1. Setup

**Objective:** import PyTorch, torchvision transforms, and our local data utilities.

**What this code does:** finds the project root, adds `src` to Python's import path, and prints the PyTorch version.

**Expected output:** PyTorch should import successfully and `Device` should be `cpu`.

**Common errors:**

- `ModuleNotFoundError: No module named 'torch'`: install dependencies with `pip install -r requirements.txt`.
- `ModuleNotFoundError: No module named 'spectiqai'`: make sure this notebook is inside the project `notebooks` folder.

In [1]:
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import transforms

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from spectiqai.data import (
    KidneyImageDataset,
    collect_image_records,
    count_by_class,
    save_split_csv,
    stratified_split,
    verify_readable_images,
)

DATASET_DIR = PROJECT_ROOT / "dataset"
RESULTS_DIR = PROJECT_ROOT / "results"
DEVICE = torch.device("cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

Project root: c:\Users\smart\Coding World\SpectiqAI
Dataset directory: c:\Users\smart\Coding World\SpectiqAI\dataset
PyTorch version: 2.12.1+cpu
Device: cpu


**How to verify success:** the cell should print paths, PyTorch version, and `Device: cpu`.

**What the result means:** your notebook can use PyTorch and the local project code.

**Next logical step:** collect image paths and labels.

## 2. Collect Image Paths and Labels

**Objective:** build a list where every image has a path, class name, and numeric label.

**What this code does:** reads filenames from the verified class folders without loading all image pixels into memory.

**Expected output:** 10000 total records, with 5000 normal and 5000 tumor.

**Common errors:**

- `FileNotFoundError`: the dataset folder is missing or class folder names changed.
- Zero records: images may be nested deeper than expected.

In [2]:
records = collect_image_records(DATASET_DIR)

print(f"Total records: {len(records)}")
print(f"Class counts: {count_by_class(records)}")
print("First record:")
print(records[0])

Total records: 10000
Class counts: {'Normal': 5000, 'Tumor': 5000}
First record:
ImageRecord(path=WindowsPath('c:/Users/smart/Coding World/SpectiqAI/dataset/Hyperspectral_normal_images/Hyperspectral_kidney_normal_0001.jpg'), class_name='Normal', label=0)


**How to verify success:** the class counts should match Phase 1.

**What the result means:** the model will later receive labels `0` for Normal and `1` for Tumor.

**Next logical step:** quickly verify that all images are still readable before splitting.

## 3. Recheck Readable Images

**Objective:** avoid putting corrupted files into training, validation, or test sets.

**What this code does:** opens each image header one by one and reports unreadable files.

**Expected output:** `Unreadable images: 0`.

**Common errors:**

- If unreadable images appear, remove them from `records` before training.
- This step may take a few seconds because there are 10000 files.

In [3]:
unreadable_images = verify_readable_images(records)
print(f"Unreadable images: {len(unreadable_images)}")

for record, error_message in unreadable_images[:10]:
    print(f"{record.path.name}: {error_message}")

if unreadable_images:
    bad_paths = {record.path for record, _ in unreadable_images}
    records = [record for record in records if record.path not in bad_paths]
    print(f"Records after removing unreadable images: {len(records)}")

Unreadable images: 0


**How to verify success:** unreadable image count should be zero.

**What the result means:** every selected image can safely be loaded by the Dataset class.

**Next logical step:** split the dataset.

## 4. Train/Validation/Test Split

**Objective:** split images into separate groups for learning, tuning, and final evaluation.

**What this code does:** uses a 70% train, 15% validation, and 15% test split while preserving class balance.

**Expected output:** about 7000 train, 1500 validation, and 1500 test images.

**Common errors:**

- Do not train on validation or test data. That would make evaluation unfair.
- Keep the random seed fixed so results are reproducible.

In [4]:
train_records, val_records, test_records = stratified_split(
    records,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
)

splits = {
    "train": train_records,
    "validation": val_records,
    "test": test_records,
}

for split_name, split_records in splits.items():
    print(f"{split_name}: {len(split_records)} images -> {count_by_class(split_records)}")

train: 7000 images -> {'Normal': 3500, 'Tumor': 3500}
validation: 1500 images -> {'Normal': 750, 'Tumor': 750}
test: 1500 images -> {'Normal': 750, 'Tumor': 750}


**How to verify success:** each split should contain equal Normal and Tumor counts.

**What the result means:** class balance is preserved, so evaluation metrics will be easier to interpret.

**Next logical step:** save the split list so later phases use the same images.

## 5. Save Split CSV

**Objective:** make the split reproducible across notebooks.

**What this code does:** writes a CSV file with split name, image path, class name, and label.

**Expected output:** a file path ending in `results/dataset_splits.csv`.

**Common errors:**

- Permission error: close the CSV if it is open in Excel.
- Missing `results` folder: the code creates it automatically.

In [5]:
split_csv_path = RESULTS_DIR / "dataset_splits.csv"
save_split_csv(splits, split_csv_path, project_root=PROJECT_ROOT)

print(f"Saved split CSV: {split_csv_path}")
print(f"File exists: {split_csv_path.exists()}")

Saved split CSV: c:\Users\smart\Coding World\SpectiqAI\results\dataset_splits.csv
File exists: True


**How to verify success:** `File exists` should be `True`.

**What the result means:** Phase 3 can reuse exactly the same train/validation/test split.

**Next logical step:** define image preprocessing and augmentation.

## 6. Preprocessing and Augmentation

**Objective:** convert images into tensors of the same size.

**What this code does:** resizes every image to 224 x 224 and converts it to a PyTorch tensor. Training images get light augmentation.

**Expected output:** printed transform pipelines.

**Common errors:**

- Do not apply random augmentation to validation or test data.
- Avoid aggressive medical-image augmentation unless a domain expert confirms it is valid.

In [6]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

print(train_transform)
print(eval_transform)

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
)
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
)


**How to verify success:** both transform pipelines should print without errors.

**What the result means:** all images will become tensors with shape `[3, 224, 224]`.

**Next logical step:** create PyTorch Dataset objects.

## 7. Create PyTorch Datasets

**Objective:** create objects that PyTorch can index like a list.

**What this code does:** wraps each split in `KidneyImageDataset`, which loads images only when needed.

**Expected output:** dataset lengths should match the split sizes.

**Common errors:**

- Empty dataset: earlier splitting failed or records were not collected.
- Runtime image error: a file became unreadable after Phase 1.

In [7]:
train_dataset = KidneyImageDataset(train_records, transform=train_transform)
val_dataset = KidneyImageDataset(val_records, transform=eval_transform)
test_dataset = KidneyImageDataset(test_records, transform=eval_transform)

print(f"Train dataset: {len(train_dataset)} images")
print(f"Validation dataset: {len(val_dataset)} images")
print(f"Test dataset: {len(test_dataset)} images")

Train dataset: 7000 images
Validation dataset: 1500 images
Test dataset: 1500 images


**How to verify success:** lengths should be 7000, 1500, and 1500 for this balanced dataset.

**What the result means:** PyTorch can now access images and labels split by purpose.

**Next logical step:** create DataLoaders for batching.

## 8. Create DataLoaders

**Objective:** load images in small batches instead of one by one manually.

**What this code does:** creates DataLoaders with batch size 16 and `num_workers=0`, which is safest for Windows and Jupyter.

**Expected output:** DataLoader batch counts for train, validation, and test.

**Common errors:**

- Memory error: reduce `BATCH_SIZE` from 16 to 8.
- Windows multiprocessing issue: keep `NUM_WORKERS = 0` in notebooks.

In [8]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 438
Validation batches: 94
Test batches: 94


**How to verify success:** the train loader should have more batches than validation and test.

**What the result means:** training can now process data in manageable chunks.

**Next logical step:** inspect one batch.

## 9. Verify Tensor Shapes

**Objective:** confirm that images and labels have the format a CNN expects.

**What this code does:** loads one batch and prints image tensor shape, label tensor shape, data type, and label values.

**Expected output:** image shape should be `[16, 3, 224, 224]` and labels should be `[16]`.

**Common errors:**

- Shape `[16, 224, 224, 3]`: channels are in the wrong position.
- Shape is too large: resize transform was not applied.

In [9]:
images, labels = next(iter(train_loader))

print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")
print(f"Image dtype: {images.dtype}")
print(f"Label dtype: {labels.dtype}")
print(f"Image value range: min={images.min().item():.4f}, max={images.max().item():.4f}")
print(f"Labels in this batch: {labels.tolist()}")

Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
Image dtype: torch.float32
Label dtype: torch.int64
Image value range: min=0.0000, max=1.0000
Labels in this batch: [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1]


**How to verify success:** the image tensor should be `[16, 3, 224, 224]`, values should be between 0 and 1, and labels should contain only 0 and 1.

**What the result means:** Phase 2 is complete. The data is ready for a CNN.

**Next logical step:** Phase 3: build a simple CNN from scratch and train it.